# Novel load and extraction

In [ ]:
!pip install -q --upgrade keras-hub
!pip install -q --upgrade keras

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 68.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 94.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
keras-nlp 0.21.1 requires keras-hub==0.21.1, but you have keras-hub 0.26.0 which is incompatible.


In [ ]:
import os
import keras_hub
import keras

import tensorflow as tf
import tensorflow.data as tf_data
import tensorflow.strings as tf_strings

In [ ]:
def extract_text(novel_file):
  start_phrase = "The Adventure of “The Western Star”"
  end_phrase = "THE END"

  with open(novel_file, "r", encoding="utf-8") as f:
      text = f.read()

  # Find exact positions
  first_start_index = text.find(start_phrase)
  if first_start_index == -1:
      raise ValueError("First occurance of the start phrase not found exactly as written.")

  second_start_index = text.find(start_phrase, first_start_index + len(start_phrase))
  if second_start_index == -1:
      raise ValueError("Secound occurance of the start phrase not found exactly as written.")

  end_index = text.find(end_phrase, second_start_index)
  if end_index == -1:
      raise ValueError("End phrase not found after start phrase.")

  # Include the end phrase itself
  end_index += len(end_phrase)

  # Slice the text
  extracted = text[second_start_index:end_index]

  return extracted

In [ ]:
txt_file_name = "61262-0.txt"
txt_file_path = os.path.join("/content", txt_file_name)

if(os.path.exists(txt_file_path)):
  print("Openning and Extracting Text...")
  novel_data = extract_text(txt_file_path)
  print("Text found and stored")
else:
  print("File not found, please upload into session")

In [ ]:
# Settings and Hyperparameters
BATCH_SIZE = 64
MIN_STRING_LENGTH = 512
SEQ_LENGTH = 128
EMBED_DIM = 256
FEED_FORWARD_DIM = 128
NUM_HEADS = 3
NUM_LAYERS = 2
VOCAB_SIZE = 5000
EPOCHS = 500
NUM_TOKENS_TO_GENERATE = 80

In [ ]:
from math import floor

def split_training_val_sets(novel_data, val_split=0.1):
  split_index = floor(len(novel_data) * (1 - val_split))
  training = novel_data[:split_index]
  validation = novel_data[split_index:]

  return training, validation

In [ ]:
raw_train_df, raw_val_df = split_training_val_sets(novel_data)

# Split the large strings into lists of smaller strings (e.g., paragraphs)
# Each element in the list will become an individual element in the dataset.
train_text_samples = raw_train_df.split("\n\n")
val_text_samples = raw_val_df.split("\n\n")

batched_train_df = (tf_data.Dataset.from_tensor_slices(train_text_samples)
                    .filter(lambda x: tf_strings.length(x) > MIN_STRING_LENGTH)
                    .batch(BATCH_SIZE, drop_remainder=True)
                    .shuffle(buffer_size=10000))

batched_val_df = (tf_data.Dataset.from_tensor_slices(val_text_samples)
                  .filter(lambda x: tf_strings.length(x) > MIN_STRING_LENGTH)
                  .batch(BATCH_SIZE, drop_remainder=True))

# Language Model

From - https://keras.io/examples/generative/text_generation_gpt/

## Tokenize Vocabulary

In [ ]:
vocab = keras_hub.tokenizers.compute_word_piece_vocabulary(
    batched_train_df,
    vocabulary_size=VOCAB_SIZE,
    lowercase=True,
    reserved_tokens=["[PAD]", "[UNK]", "[BOS]"]
)
actual_vocab_size = len(vocab)
print(f"Actual vocabulary size derived from data: {actual_vocab_size}")

Actual vocabulary size derived from data: 390


In [ ]:
tokenizer = keras_hub.tokenizers.WordPieceTokenizer(
    vocabulary=vocab,
    sequence_length=SEQ_LENGTH,
    lowercase=True
)

In [ ]:
start_packer = keras_hub.layers.StartEndPacker(
    sequence_length = SEQ_LENGTH,
    start_value=tokenizer.token_to_id("[BOS]")
)

def preprocessor(inputs):
  outputs = tokenizer(inputs)
  features = start_packer(outputs)
  labels = outputs
  return features, labels

train_ds = batched_train_df.map(preprocessor, num_parallel_calls=tf_data.AUTOTUNE).prefetch(tf_data.AUTOTUNE)
val_ds = batched_val_df.map(preprocessor, num_parallel_calls=tf_data.AUTOTUNE).prefetch(tf_data.AUTOTUNE)

In [ ]:
import pickle

# Save the tokenizer
with open('word_piece_tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)

print("Tokenizer saved successfully as 'word_piece_tokenizer.pkl'")

Tokenizer saved successfully as 'word_piece_tokenizer.pkl'


## Build Model

In [ ]:
inputs = keras.layers.Input(shape=(None,), dtype="int32")

embedding_layer = keras_hub.layers.TokenAndPositionEmbedding(
    vocabulary_size=actual_vocab_size, # Use actual_vocab_size here
    sequence_length=SEQ_LENGTH,
    embedding_dim=EMBED_DIM,
    mask_zero=True
)
x = embedding_layer(inputs) # Apply the embedding layer to the inputs

for _ in range(NUM_LAYERS):
  decoder_layer = keras_hub.layers.TransformerDecoder(
      num_heads=NUM_HEADS,
      intermediate_dim=FEED_FORWARD_DIM
  )
  x = decoder_layer(x)

outputs = keras.layers.Dense(actual_vocab_size)(x) # Use actual_vocab_size here
model = keras.Model(inputs, outputs)
loss = keras.losses.SparseCategoricalCrossentropy(from_logits=True)
perplexity = keras_hub.metrics.Perplexity(from_logits=True)

model.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss=loss,
    metrics=[perplexity]
)
model.summary()

Model: "functional_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_7 (InputLayer)      │ (None, None)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ token_and_position_embedding_7  │ (None, None, 256)      │       132,608 │
│ (TokenAndPositionEmbedding)     │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_decoder_13          │ (None, None, 256)      │       329,085 │
│ (TransformerDecoder)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_decoder_14          │ (None, None, 256)      │       329,085 │
│ (TransformerDecoder)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, None, 390)      │       100,230 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 891,008 (3.40 MB)

 Trainable params: 891,008 (3.40 MB)

 Non-trainable params: 0 (0.00 B)

## Training the model


In [ ]:
model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS)

Epoch 1/500


/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:982: UserWarning: Layer 'position_embedding' (of type PositionEmbedding) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:982: UserWarning: Layer 'query' (of type EinsumDense) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:982: UserWarning: Layer 'key' (of type EinsumDense) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(
/usr/local/lib/python3.

      1/Unknown 9s 9s/step - loss: 6.3290 - perplexity: 560.6008

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


1/1 ━━━━━━━━━━━━━━━━━━━━ 9s 9s/step - loss: 6.3290 - perplexity: 560.6008
Epoch 2/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 433ms/step - loss: 6.3231 - perplexity: 557.2980
Epoch 3/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 444ms/step - loss: 6.3172 - perplexity: 554.0146
Epoch 4/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 431ms/step - loss: 6.3113 - perplexity: 550.7520
Epoch 5/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 431ms/step - loss: 6.3054 - perplexity: 547.5110
Epoch 6/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 425ms/step - loss: 6.2995 - perplexity: 544.2927
Epoch 7/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 431ms/step - loss: 6.2936 - perplexity: 541.0958
Epoch 8/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 455ms/step - loss: 6.2877 - perplexity: 537.9202
Epoch 9/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 453ms/step - loss: 6.2818 - perplexity: 534.7661
Epoch 10/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 440ms/step - loss: 6.2760 - perplexity: 531.6332
Epoch 11/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 435ms/step - loss: 6.2701 - perplexity: 528.5209
Epoch 12/500
1/1 ━━━━━━━━━━━━━━━━━

In [ ]:
# Save the model
model.save('text_generation_model.keras')

print("Model saved successfully as 'text_generation_model.keras'")

Model saved successfully as 'text_generation_model.keras'


## Test for demo

In [ ]:
from keras.preprocessing.sequence import pad_sequences
import numpy as np
import pickle

with open("word_piece_tokenizer.pkl", "rb") as f:
    tokenizer = pickle.load(f)

from keras.models import load_model
model = load_model("text_generation_model.keras")

In [ ]:
def generate_text(model, tokenizer, seed_text, max_length, num_tokens_to_generate=100, temperature=0.1):
    result = []
    text = seed_text

    for _ in range(num_tokens_to_generate):
      # Encode the current text input
      encoded = tokenizer([text])

      # Get model predictions (logits) for the entire sequence
      # predictions shape: (1, SEQ_LENGTH, VOCAB_SIZE)
      predictions = model.predict(encoded, verbose=0)

      # Take the logits for the last token in the sequence (the one to predict)
      # next_token_logits shape: (VOCAB_SIZE,)
      next_token_logits = predictions[0, -1, :]

      # Apply temperature scaling to the logits
      next_token_logits = next_token_logits / temperature

      # Convert logits to probabilities using softmax
      probabilities = tf.nn.softmax(next_token_logits).numpy()

      # Sample the next token index based on the probabilities
      next_index = np.random.choice(len(probabilities), p=probabilities)
      next_word = tokenizer.id_to_token(next_index)

      # Append the generated word to the result
      result.append(next_word)

      # Update the text for the next iteration
      # The tokenizer will handle truncation/padding to SEQ_LENGTH
      text += " " + next_word

    return seed_text + " " + " ".join(result)

In [ ]:
text_file_input = input("Please input the file name for the input")
if os.path.exists(text_file_input):
  with open(text_file_input, "r", encoding="utf-8") as f:
    seed_text = f.read()
else:
  print("File not found, using default prompt")
  seed_text = "The Adventure of “The Western Star”"

tokens_to_generate_str = input("Please input n for the generated sequence")
if tokens_to_generate_str.isdigit():
   tokens_to_generate = int(tokens_to_generate_str)
else:
   tokens_to_generate = NUM_TOKENS_TO_GENERATE

print(generate_text(
    model=model,
    tokenizer=tokenizer,
    seed_text=seed_text,
    max_length=SEQ_LENGTH,
    num_tokens_to_generate=tokens_to_generate,
    temperature=0.8
))

File not found, using default prompt
Please input n for the generated sequence20
The Adventure of “The Western Star” c with ##u brought ##a it _ black ##u ##ived ##c m prime ##ic ##n ##c the ##ct ##ce the


## Mini Report and Evaluation for model

This model was a very simple one heavily inspired by the documentation of lanugage models included in keras guides. The design was very rough but followed this methodology:
- Firstly, take the text file and split and strip anything that isn't directly the story or its chapters.
- Secondly, split the text file into a train and test lists and batch the data for a the model.
- Thirdly, build and train a tokenizer to take the input and parse into structure token of certain sizes, as well as establishing the vocabulary for the model.
- Fourthly, embed the tokens as an input layers into our model and use a Decoder Transformer to decode our embedded layers depending on the tokens and number of layers.
- Finally, add a dense layer to condense the decoder transformer into the vocabulary of the model.

We built the model and fit it over 500 epochs to get a slow improvement on our perplexity and loss using a small learning rate.

Comparing our output to that of another LLM like Chat-GPT, shows that our model is clearly very flawed and require additional work. The main issue is the punctuation making the words unreadible, and because they are unreadible, despite the readible words we cannot infer any sort of structure. Contrasting this a Chat-GPT output which although is very generic and vague still has very good and understandble structure with no spelling mistakes or hallucinated punctuation.

What could be improved to make the model better?
- More preprocessing to handle punctuation and normalise whitespace.
- More time spent tuning hyperparameters and experimenting with layers.

What does Chat-GPT potentially do to make their model better?
- More data to train the model on, increasing the models knowledge base.
- More sophisticated model design and complexity of network to cover more bases.

The only upside over my model than Chat-GPT is the environmental cost, as models like Chat-GPT and Gemmani have intense and resource hungry taining processes.